In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
rl = "200000"
model = PPO.load(f"RLmodels/ppo_2566_{rl}_steps.zip", device="cpu")
model

In [3]:
fragment_pool = pd.read_csv("source/SMILES_list.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=5,
                          max_k=5,
                          max_steps=6,
                          eps=1e-8,
                          r_cap=100,
                          shaping=None,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        #print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [4]:
df = generate_SMILES(model, eval_env, n_episodes=10)
df.to_csv(f"generated_OPSs/rl_candidates_{rl}_noshaping.csv", index=False)
df

,episode,smiles,selected_fragments,reward
0,0,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
1,1,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
2,2,N#Cc1cnc2cc(O)c(N)cc2c1C(=O)NN1C(=O)C(Cl)C1C(=...,"(N#Cc1cnc2cc(O)c(N)cc2c1, O=CNN1CC(Cl)C1=O, O=...",28.017084
3,3,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
4,4,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
5,5,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
6,6,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
7,7,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
8,8,N#Cc1cnc2cc(O)c(N)cc2c1C(=O)NN1C(=O)C(Cl)C1C(=...,"(N#Cc1cnc2cc(O)c(N)cc2c1, O=CNN1CC(Cl)C1=O, O=...",28.017084
9,9,N#Cc1cnc2cc(O)c(N)cc2c1-c1c(C#N)cnc2cc(O)c(N)c...,"(N#Cc1cnc2cc(O)c(N)cc2c1, N#Cc1cnc2cc(O)c(N)cc...",56.728916
